# تطبيق اليوم الأول — شاهد أنواع تعلم الآلة على بيانات منافذ
شغّل الخلايا بالترتيب واقرأ الشرح. جميع الأكواد مكتملة، ولا توجد كتابة أو تسليم مطلوب.

## 1) تحميل البيانات
سنحمّل بيانات منافذ تلقائيًا. كل صف يمثل عميلًا، وكل عمود يمثل معلومة عنه.

In [ ]:
import pandas as pd
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'
df = pd.read_csv(DATA_URL)
print('تم تحميل البيانات ✅')
print('عدد الصفوف والأعمدة:', df.shape)
display(df.head(5))

**كيف نقرأ الناتج؟** الرقم الأول في `shape` هو عدد العملاء، والثاني عدد الأعمدة. في `churned_30d`: القيمة 1 تعني أن العميل توقف خلال 30 يومًا، و0 تعني أنه لم يتوقف.

## 2) مثال تصنيف (Classification)
السؤال: هل سيتوقف العميل أم لا؟ لأن الإجابة فئة 0 أو 1، فهذه مسألة تصنيف.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
features = ['orders_per_month','avg_basket_sar','days_since_last_order','promo_usage_rate','avg_rating']
X = df[features]
y = df['churned_30d']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
classifier = Pipeline([('missing', SimpleImputer(strategy='median')), ('model', DecisionTreeClassifier(max_depth=3, random_state=42))])
classifier.fit(X_train, y_train)
class_prediction = classifier.predict(X_test)
display(pd.DataFrame({'الحقيقة':y_test.iloc[:10].values, 'توقع النموذج':class_prediction[:10]}))

**كيف نقرأ الناتج؟** كل صف يعرض الحقيقة ثم توقع النموذج. إذا تطابق الرقمان كان التوقع صحيحًا، وإذا اختلفا أخطأ النموذج. المخرج هنا فئة، وليس رقمًا مستمرًا.

## 3) مثال انحدار (Regression)
السؤال: ما قيمة سلة العميل بالريال؟ لأن الإجابة رقم يمكن أن يأخذ قيمًا كثيرة، فهذه مسألة انحدار.

In [ ]:
from sklearn.linear_model import LinearRegression
reg_features = ['orders_per_month','days_since_last_order','promo_usage_rate','tenure_months','avg_rating']
X_reg = df[reg_features]
y_reg = df['avg_basket_sar']
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)
regressor = Pipeline([('missing', SimpleImputer(strategy='median')), ('model', LinearRegression())])
regressor.fit(Xr_train, yr_train)
number_prediction = regressor.predict(Xr_test)
display(pd.DataFrame({'الحقيقة بالريال':yr_test.iloc[:10].round(2).values, 'توقع النموذج':number_prediction[:10].round(2)}))

**كيف نقرأ الناتج؟** التوقع هنا قيمة بالريال. الفرق بين الحقيقة والتوقع هو خطأ النموذج في تلك الحالة. سنتعلم قياس هذا الخطأ بالتفصيل لاحقًا.

## 4) مثال تعلم غير خاضع للإشراف — K-Means
هنا لا نستخدم هدفًا y. نعطي الخوارزمية خصائص العملاء، وهي تجمع العملاء المتشابهين.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
cluster_features = ['orders_per_month','avg_basket_sar','days_since_last_order','promo_usage_rate']
cluster_prep = Pipeline([('missing', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
Z = cluster_prep.fit_transform(df[cluster_features])
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df_demo = df.copy()
df_demo['cluster'] = kmeans.fit_predict(Z)
profile = df_demo.groupby('cluster')[cluster_features].mean().round(2)
display(profile)

**كيف نقرأ الناتج؟** كل صف يمثل مجموعة اكتشفتها الخوارزمية، والأرقام هي متوسط خصائص عملائها. رقم المجموعة مثل 0 أو 1 لا يحمل معنى جاهزًا؛ نفهم المجموعة من خصائصها ثم نسميها.

## 5) الخلاصة

In [ ]:
summary = pd.DataFrame([
 {'الطريقة':'Classification','هل يوجد y؟':'نعم','المخرج':'فئة 0 أو 1','المثال':'هل سيتوقف العميل؟'},
 {'الطريقة':'Regression','هل يوجد y؟':'نعم','المخرج':'رقم','المثال':'قيمة السلة المتوقعة'},
 {'الطريقة':'K-Means Clustering','هل يوجد y؟':'لا','المخرج':'رقم مجموعة','المثال':'شرائح عملاء متشابهة'}
])
display(summary)

لقد شاهدت الآن الفرق عمليًا: التصنيف يعطي فئة، والانحدار يعطي رقمًا، والتجميع يكتشف مجموعات من دون إجابة صحيحة جاهزة. في الأيام التالية سنتعلم كيف بُنيت هذه الخطوات بالتدرج.